# D3/D4 Hands-on: CNN Classification

2026 Winter

本 Notebook 會自動設定環境、安裝依賴、並執行完整的訓練流程。

支援環境：**Google Colab** / **Kaggle** / **本機**

支援任務：
- **D3**: CT 影像肝臟腫瘤分類 (Liver CT)
- **D4**: 胸部 X 光分類 (Chest X-ray Hackathon)

In [ ]:
# Step 0: Select task
TASK = "d3_liver_ct"  #@param ["d3_liver_ct", "d4_hackathon"]
GROUP_NUM = 1          #@param {type:"slider", min:1, max:15}

print(f"Selected task: {TASK}")
if TASK == "d4_hackathon":
    print(f"Group number: {GROUP_NUM}")

In [ ]:
# Step 1: Detect environment
import sys, os
try:
    import google.colab
    ENV = "colab"
except ImportError:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        ENV = "kaggle"
    else:
        ENV = "local"

print(f"Detected environment: {ENV}")

In [ ]:
# Step 2: Mount Drive (Colab only)
if ENV == "colab":
    from google.colab import drive
    drive.mount("drive", force_remount=True)
    print("Drive mounted.")
else:
    print("Drive mount skipped (not Colab).")

In [ ]:
# Step 3: Clone repo
REPO_URL = "https://github.com/cbe135/d3-hands-on-liver-classification.git"
REPO_NAME = "d3-hands-on-liver-classification"

if os.path.exists(REPO_NAME):
    print(f"Repo already exists at ./{REPO_NAME}")
else:
    !git clone $REPO_URL
    print("Repo cloned.")

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Step 4: Install dependencies
if ENV == "local":
    !uv sync
else:
    !pip install -r requirements.txt
print("Dependencies installed.")

In [ ]:
# Step 5: Import libraries
import logging
import math
import os
import shutil
import yaml
import zipfile
from datetime import datetime
from tqdm.notebook import tqdm

import monai
import numpy as np
import timm
import torch
from matplotlib import pyplot as plt
from monai.data import CacheDataset, DataLoader, Dataset
from monai.transforms import (
    Compose, EnsureTyped, LoadImaged,
    MaskIntensityd, RandAffined, RandFlipd,
    RepeatChanneld, Resized, ScaleIntensityd,
    ScaleIntensityRanged, Transform,
)
from monai.utils.misc import set_determinism
from sklearn.metrics import auc, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split

print(f"MONAI: {monai.__version__}, PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
# Step 6: Configure (task-aware defaults)
from src.main import get_default_args

args = get_default_args(TASK, GROUP_NUM)

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s.%(msecs)03d][%(levelname)5s](%(name)s) - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)
logger = logging.getLogger()
set_determinism(args["environ"]["seed"])
print(f"Task: {args['task']}, Data: {args['environ']['data_name']}")

In [ ]:
# Step 7: Setup data
from src.env_setup import setup_data, get_data_count

setup_data(args)
get_data_count(args)

In [ ]:
# Step 8: Load and split data
from src.data import load_data_list, populate_data_lists

data_dicts = load_data_list(args)
train_dicts, val_dicts, test_dicts = populate_data_lists(args, data_dicts)

print(f"{len(train_dicts)} data for training")
print(f"{len(val_dicts)} data for validation")
print(f"{len(test_dicts)} data for testing")

In [ ]:
# Step 9: Build transforms and datasets
from src.transforms import build_train_transform, build_val_transform
from src.data import generate_dataset, generate_dataloader

train_transform = build_train_transform(args)
val_transform = build_val_transform(args)

train_set = generate_dataset(args, train_dicts, train_transform)
val_set = generate_dataset(args, val_dicts, val_transform)
test_set = generate_dataset(args, test_dicts, val_transform)

print("Datasets ready.")

In [ ]:
# Step 10: Train
from src.train import train_pipeline
from src.utils import plot_loss_curves

print("Starting training...")
model, train_loader, val_loader, record = train_pipeline(args, train_set, val_set)
plot_loss_curves(args, record)
print("Training complete!")

In [ ]:
# Step 11: Evaluate
from src.evaluate import infer, plot_roc_and_show_result

best_state = torch.load("best_weights.pth", weights_only=True)
model.load_state_dict(best_state)

train_true, train_pred = infer(args, model, train_loader, True)
val_true, val_pred = infer(args, model, val_loader, True)
test_loader = generate_dataloader(args, test_set)
test_true, test_pred = infer(args, model, test_loader, True)

plot_roc_and_show_result(args, train_true, train_pred, title="Train")
plot_roc_and_show_result(args, val_true, val_pred, title="Validation")
plot_roc_and_show_result(args, test_true, test_pred, title="Test")

print("Evaluation complete!")

In [ ]:
# Step 12: Grad-CAM (optional)
from src.evaluate import grad_cam

# Update the path below based on your task
if TASK == "d3_liver_ct":
    grad_cam(model, "/content/liver_data/images/liver_118_13.nii.gz", 0, args=args)
elif TASK == "d4_hackathon":
    # Provide a sample image path from the D4 dataset
    print("Provide a sample image path for Grad-CAM visualization.")